## Code Execution Order

1. first run `abs_raw_data_get.py`, which retrieves the ABS raw Excel file from the official website, and then `abs_raw_data_clean.py`, which cleans the workbook, exports the processed CSV files, and loads them into DuckDB. Next, run `nger_raw_data_get.py` to retrieve the yearly NGER raw CSV files, followed by `nger_raw_data_merge.py` to combine the multi-year files into one dataset, and then `nger_raw_date_clean.py` to standardize columns, clean values, and store the final NGER table in DuckDB. Finally, run `renewable_raw_data_get.py` to retrieve the renewable project CSV files, and `renewable_raw_date_clean.py` to clean, integrate, and load the renewable datasets into DuckDB.

2. data_cleaning_integration_augmentation.ipynb  
   - Performs data cleaning and integration, updating assignment_1.duckdb.

3. 3 Analysis+Visualisation.ipynb
   - Performs data exploration and generates visualisations using assignment_1.duckdb.

All scripts should be executed in the above order.

## Usage

This notebook should be run after the data acquisition and preprocessing step. It expects an existing DuckDB database(assignment_1.duckdb) in the same working directory, with the required base tables already loaded: `abs_economy`, `abs_industry`, `nger_facility_emissions`, `renewable_projects_accredited`, `renewable_projects_combined`, `renewable_projects_committed`, and `renewable_projects_probable`.

Running this notebook will perform data integration, state-year aggregation, geocoding augmentation, and write the enhanced results back to DuckDB.

In [1]:
import duckdb
con = duckdb.connect("assignment_1.duckdb")
print(con.execute("SHOW TABLES").fetchall())

[('abs_economy',), ('abs_industry',), ('nger_facility_emissions',), ('renewable_projects_accredited',), ('renewable_projects_combined',), ('renewable_projects_committed',), ('renewable_projects_probable',)]


In [2]:
con.execute("""
DROP TABLE IF EXISTS abs_industry;
""")

print(con.execute("SHOW TABLES").fetchdf())

                            name
0                    abs_economy
1        nger_facility_emissions
2  renewable_projects_accredited
3    renewable_projects_combined
4   renewable_projects_committed
5    renewable_projects_probable


In [3]:
con.execute("""
ALTER TABLE abs_economy RENAME TO abs_economy_industry;
""")

print(con.execute("SHOW TABLES").fetchdf())

                            name
0           abs_economy_industry
1        nger_facility_emissions
2  renewable_projects_accredited
3    renewable_projects_combined
4   renewable_projects_committed
5    renewable_projects_probable


In [4]:
print(con.execute("DESCRIBE renewable_projects_combined").fetchdf())

                    column_name column_type null   key default extra
0            Accreditation code     VARCHAR  YES  None    None  None
1            Power station name     VARCHAR  YES  None    None  None
2                         State     VARCHAR  YES  None    None  None
3                      Postcode      DOUBLE  YES  None    None  None
4       Installed capacity (MW)      DOUBLE  YES  None    None  None
5               Fuel Source (s)     VARCHAR  YES  None    None  None
6      Accreditation start date     VARCHAR  YES  None    None  None
7                 Approval date     VARCHAR  YES  None    None  None
8                project_status     VARCHAR  YES  None    None  None
9                  Project Name     VARCHAR  YES  None    None  None
10                       State      VARCHAR  YES  None    None  None
11                  MW Capacity      DOUBLE  YES  None    None  None
12                  Fuel Source     VARCHAR  YES  None    None  None
13  Committed Date (Month/Year)   

In [5]:
print(con.execute("""
SELECT 'accredited' AS source, COUNT(*) AS cnt FROM renewable_projects_accredited
UNION ALL
SELECT 'committed' AS source, COUNT(*) AS cnt FROM renewable_projects_committed
UNION ALL
SELECT 'probable' AS source, COUNT(*) AS cnt FROM renewable_projects_probable
UNION ALL
SELECT 'combined' AS source, COUNT(*) AS cnt FROM renewable_projects_combined
""").fetchdf())

       source  cnt
0  accredited   36
1   committed   36
2    probable   59
3    combined  131


In [6]:
print(con.execute("""
SELECT * 
FROM renewable_projects_combined
LIMIT 10
""").fetchdf())

  Accreditation code                                 Power station name State  \
0           SRPXQLO8                      DONS FORT - SOLAR W SGU - QLD   QLD   
1           SRPYNSK0                    Mintus Coles - Solar w SGU- NSW   NSW   
2           SRPXQLP1               Ventora - Acacia Ridge - Solar - QLD   QLD   
3           SRPYNSK3         Bayside Aged Care - Solar - Morisset - NSW   NSW   
4           SRPVWAQ0  Matrix Composites and Engineering Pty Ltd - So...    WA   
5           SRPXQLP4                     Accensi Narangba - Solar - QLD   QLD   
6           SRPXSA19                      Bunge Port Giles - Solar - SA    SA   
7           SRPVWAP2  HBF Stadium Carpark 4 PV System - Solar w SGU ...    WA   
8           SRPYNSJ2          Stockland M-park Building C - Solar - NSW   NSW   
9           SRPXVCU9               Waverley Gardens - Solar w SGU - VIC   VIC   

   Postcode  Installed capacity (MW) Fuel Source (s) Accreditation start date  \
0    4660.0                

In [7]:
print(con.execute("""
SELECT project_status, COUNT(*) AS cnt
FROM renewable_projects_combined
GROUP BY project_status
ORDER BY project_status
""").fetchdf())

  project_status  cnt
0     accredited   36
1      committed   36
2       probable   59


In [8]:
print(con.execute("""
SELECT
    project_status,
    COUNT(*) AS total_rows,
    COUNT("Power station name") AS power_station_name_non_null,
    COUNT("Project Name") AS project_name_non_null,
    COUNT("Probable Projects") AS probable_projects_non_null,
    COUNT("Installed capacity (MW)") AS installed_capacity_non_null,
    COUNT("MW Capacity") AS mw_capacity_non_null,
    COUNT("Fuel Source (s)") AS fuel_source_s_non_null,
    COUNT("Fuel Source") AS fuel_source_non_null
FROM renewable_projects_combined
GROUP BY project_status
ORDER BY project_status
""").fetchdf())

  project_status  total_rows  power_station_name_non_null  \
0     accredited          36                           36   
1      committed          36                            0   
2       probable          59                            0   

   project_name_non_null  probable_projects_non_null  \
0                      0                           0   
1                     36                           0   
2                      0                          59   

   installed_capacity_non_null  mw_capacity_non_null  fuel_source_s_non_null  \
0                           36                     0                      36   
1                            0                    36                       0   
2                            0                    56                       0   

   fuel_source_non_null  
0                     0  
1                    36  
2                    59  


In [9]:
print(con.execute("DESCRIBE renewable_projects_accredited").fetchdf())

                column_name column_type null   key default extra
0        Accreditation code     VARCHAR  YES  None    None  None
1        Power station name     VARCHAR  YES  None    None  None
2                     State     VARCHAR  YES  None    None  None
3                  Postcode      BIGINT  YES  None    None  None
4   Installed capacity (MW)      DOUBLE  YES  None    None  None
5           Fuel Source (s)     VARCHAR  YES  None    None  None
6  Accreditation start date     VARCHAR  YES  None    None  None
7             Approval date     VARCHAR  YES  None    None  None
8            project_status     VARCHAR  YES  None    None  None


In [10]:
print(con.execute("DESCRIBE renewable_projects_committed").fetchdf())

                   column_name column_type null   key default extra
0                 Project Name     VARCHAR  YES  None    None  None
1                       State      VARCHAR  YES  None    None  None
2                  MW Capacity      DOUBLE  YES  None    None  None
3                  Fuel Source     VARCHAR  YES  None    None  None
4  Committed Date (Month/Year)     VARCHAR  YES  None    None  None
5               project_status     VARCHAR  YES  None    None  None


In [11]:
print(con.execute("DESCRIBE renewable_projects_probable").fetchdf())

         column_name column_type null   key default extra
0  Probable Projects     VARCHAR  YES  None    None  None
1             State      VARCHAR  YES  None    None  None
2        MW Capacity      DOUBLE  YES  None    None  None
3        Fuel Source     VARCHAR  YES  None    None  None
4     project_status     VARCHAR  YES  None    None  None


In [12]:
con.execute("""
CREATE OR REPLACE TABLE renewable_projects_combined_standardized AS
SELECT
    COALESCE("Power station name", "Project Name", "Probable Projects") AS project_name,
    TRIM(UPPER(COALESCE("State", "State "))) AS state,
    "Postcode" AS postcode,
    COALESCE("Installed capacity (MW)", "MW Capacity") AS capacity_mw,
    COALESCE("Fuel Source (s)", "Fuel Source") AS fuel_source,
    COALESCE("Accreditation start date", "Committed Date (Month/Year)") AS project_date,
    project_status AS status
FROM renewable_projects_combined
""")

In [13]:
print(con.execute("DESCRIBE renewable_projects_combined_standardized").fetchdf())

    column_name column_type null   key default extra
0  project_name     VARCHAR  YES  None    None  None
1         state     VARCHAR  YES  None    None  None
2      postcode      DOUBLE  YES  None    None  None
3   capacity_mw      DOUBLE  YES  None    None  None
4   fuel_source     VARCHAR  YES  None    None  None
5  project_date     VARCHAR  YES  None    None  None
6        status     VARCHAR  YES  None    None  None


In [14]:
print(con.execute("""
SELECT COUNT(*) FROM renewable_projects_combined_standardized
""").fetchdf())

   count_star()
0           131


In [15]:
print(con.execute("""
SELECT status, COUNT(*) 
FROM renewable_projects_combined_standardized
GROUP BY status
ORDER BY status
""").fetchdf())

       status  count_star()
0  accredited            36
1   committed            36
2    probable            59


In [16]:
print(con.execute("""
SELECT
    COUNT(project_name) AS name_ok,
    COUNT(state) AS state_ok,
    COUNT(capacity_mw) AS capacity_ok,
    COUNT(fuel_source) AS fuel_ok
FROM renewable_projects_combined_standardized
""").fetchdf())

   name_ok  state_ok  capacity_ok  fuel_ok
0      131       131          128      131


In [17]:
con.execute("""
DROP TABLE IF EXISTS renewable_projects_accredited;
DROP TABLE IF EXISTS renewable_projects_combined;
DROP TABLE IF EXISTS renewable_projects_committed;
DROP TABLE IF EXISTS renewable_projects_probable;
""")
print(con.execute("SHOW TABLES").fetchdf())
 

                                       name
0                      abs_economy_industry
1                   nger_facility_emissions
2  renewable_projects_combined_standardized


In [18]:
print(con.execute("DESCRIBE abs_economy_industry").fetchdf())
print(con.execute("SELECT * FROM abs_economy_industry LIMIT 10").fetchdf())

                                       column_name column_type null   key  \
0                                             Code     VARCHAR  YES  None   
1                                            Label     VARCHAR  YES  None   
2                                             Year      DOUBLE  YES  None   
3               Number of non-employing businesses      BIGINT  YES  None   
4    Number of employing businesses: 1-4 employees      BIGINT  YES  None   
..                                             ...         ...  ...   ...   
122                    Apartments - removals (no.)      BIGINT  YES  None   
123                       Apartments - total (no.)      BIGINT  YES  None   
124                 Total dwelling additions (no.)      BIGINT  YES  None   
125                  Total dwelling removals (no.)      BIGINT  YES  None   
126                          Total dwellings (no.)      BIGINT  YES  None   

    default extra  
0      None  None  
1      None  None  
2      None  No

In [19]:
print(con.execute("""
SELECT DISTINCT Code
FROM abs_economy_industry
ORDER BY Code
""").fetchdf())

                                                   Code
0                                                     1
1                                                   101
2                                                 10102
3                                             101021007
4                                             101021008
...                                                 ...
2905                                              90104
2906                                          901041004
2907                                              9OTER
2908                                                AUS
2909  Note: Main Structure and Greater Capital City ...

[2910 rows x 1 columns]


In [20]:
print(con.execute("""
SELECT DISTINCT Code, Label
FROM abs_economy_industry
ORDER BY Code
LIMIT 100
""").fetchdf())

         Code                    Label
0           1          New South Wales
1         101           Capital Region
2       10102               Queanbeyan
3   101021007                Braidwood
4   101021008                  Karabar
..        ...                      ...
95  103041079         Orange Surrounds
96        104  Coffs Harbour - Grafton
97      10401          Clarence Valley
98  104011080                  Grafton
99  104011081        Grafton Surrounds

[100 rows x 2 columns]


In [21]:
print(con.execute("""
SELECT DISTINCT Code, Label
FROM abs_economy_industry
WHERE Code IN ('1','2','3','4','5','6','7','8','9','AUS')
ORDER BY Code
""").fetchdf())

  Code                         Label
0    1               New South Wales
1    2                      Victoria
2    3                    Queensland
3    4               South Australia
4    5             Western Australia
5    6                      Tasmania
6    7            Northern Territory
7    8  Australian Capital Territory
8    9             Other Territories
9  AUS                     Australia


In [22]:
con.execute("""
CREATE OR REPLACE TABLE abs_state_only AS
SELECT
    CASE
        WHEN Code = '1' THEN 'NSW'
        WHEN Code = '2' THEN 'VIC'
        WHEN Code = '3' THEN 'QLD'
        WHEN Code = '4' THEN 'SA'
        WHEN Code = '5' THEN 'WA'
        WHEN Code = '6' THEN 'TAS'
        WHEN Code = '7' THEN 'NT'
        WHEN Code = '8' THEN 'ACT'
    END AS state,
    CAST(Year AS INTEGER) AS year,
    "Total number of businesses" AS total_businesses,
    "Number of employing businesses: 1-4 employees" AS small_businesses,
    "Total dwellings (no.)" AS total_dwellings
FROM abs_economy_industry
WHERE Code IN ('1','2','3','4','5','6','7','8')
""")

In [23]:
print(con.execute("DESCRIBE abs_state_only").fetchdf())

print(con.execute("""
SELECT state, year, COUNT(*) AS n
FROM abs_state_only
GROUP BY state, year
ORDER BY state, year
""").fetchdf())

print(con.execute("""
SELECT *
FROM abs_state_only
LIMIT 20
""").fetchdf())

        column_name column_type null   key default extra
0             state     VARCHAR  YES  None    None  None
1              year     INTEGER  YES  None    None  None
2  total_businesses      BIGINT  YES  None    None  None
3  small_businesses      BIGINT  YES  None    None  None
4   total_dwellings      BIGINT  YES  None    None  None
   state  year  n
0    ACT  2011  1
1    ACT  2016  1
2    ACT  2017  1
3    ACT  2018  1
4    ACT  2019  1
..   ...   ... ..
75    WA  2020  1
76    WA  2021  1
77    WA  2022  1
78    WA  2023  1
79    WA  2024  1

[80 rows x 3 columns]
   state  year  total_businesses  small_businesses  total_dwellings
0    NSW  2011              <NA>              <NA>             <NA>
1    NSW  2016              <NA>              <NA>             <NA>
2    NSW  2017              <NA>              <NA>          3112186
3    NSW  2018              <NA>              <NA>          3171322
4    NSW  2019              <NA>              <NA>          3239814
5    NSW  2

In [24]:
con.execute("""
DROP TABLE IF EXISTS abs_economy_industry;
ALTER TABLE abs_state_only RENAME TO abs_economy_industry;
""")
print(con.execute("SHOW TABLES").fetchdf())
print(con.execute("DESCRIBE abs_economy_industry").fetchdf())
print(con.execute("SELECT * FROM abs_economy_industry LIMIT 10").fetchdf())

                                       name
0                      abs_economy_industry
1                   nger_facility_emissions
2  renewable_projects_combined_standardized
        column_name column_type null   key default extra
0             state     VARCHAR  YES  None    None  None
1              year     INTEGER  YES  None    None  None
2  total_businesses      BIGINT  YES  None    None  None
3  small_businesses      BIGINT  YES  None    None  None
4   total_dwellings      BIGINT  YES  None    None  None
  state  year  total_businesses  small_businesses  total_dwellings
0   NSW  2011              <NA>              <NA>             <NA>
1   NSW  2016              <NA>              <NA>             <NA>
2   NSW  2017              <NA>              <NA>          3112186
3   NSW  2018              <NA>              <NA>          3171322
4   NSW  2019              <NA>              <NA>          3239814
5   NSW  2020            786246            225070          3291260
6   NSW  2021

In [25]:
con.execute("""
CREATE OR REPLACE TABLE abs_temp AS
SELECT
    state,
    year,
    total_businesses,
    small_businesses
FROM abs_economy_industry
""")

con.execute("""
DROP TABLE IF EXISTS abs_economy_industry;
ALTER TABLE abs_temp RENAME TO abs_economy_industry;
""")

In [26]:
print(con.execute("DESCRIBE abs_economy_industry").fetchdf())

print(con.execute("""
SELECT *
FROM abs_economy_industry
ORDER BY state, year
LIMIT 20
""").fetchdf())

        column_name column_type null   key default extra
0             state     VARCHAR  YES  None    None  None
1              year     INTEGER  YES  None    None  None
2  total_businesses      BIGINT  YES  None    None  None
3  small_businesses      BIGINT  YES  None    None  None
   state  year  total_businesses  small_businesses
0    ACT  2011              <NA>              <NA>
1    ACT  2016              <NA>              <NA>
2    ACT  2017              <NA>              <NA>
3    ACT  2018              <NA>              <NA>
4    ACT  2019              <NA>              <NA>
5    ACT  2020             29724              8319
6    ACT  2021             31490              9995
7    ACT  2022             33813             10867
8    ACT  2023             35086             10908
9    ACT  2024             36316             10764
10   NSW  2011              <NA>              <NA>
11   NSW  2016              <NA>              <NA>
12   NSW  2017              <NA>              <NA>
1

In [27]:
con.execute("""
CREATE OR REPLACE TABLE abs_temp AS
SELECT *
FROM abs_economy_industry
WHERE total_businesses IS NOT NULL
""")

con.execute("""
DROP TABLE IF EXISTS abs_economy_industry;
ALTER TABLE abs_temp RENAME TO abs_economy_industry;
""")

In [28]:
print(con.execute("DESCRIBE nger_facility_emissions").fetchdf())

                         column_name column_type null   key default extra
0                   reporting_entity     VARCHAR  YES  None    None  None
1                      facility_name     VARCHAR  YES  None    None  None
2                      facility_type     VARCHAR  YES  None    None  None
3                              state     VARCHAR  YES  None    None  None
4          electricity_production_gj      DOUBLE  YES  None    None  None
5         electricity_production_mwh      DOUBLE  YES  None    None  None
6                       scope1_tco2e      DOUBLE  YES  None    None  None
7                       scope2_tco2e      DOUBLE  YES  None    None  None
8              total_emissions_tco2e      DOUBLE  YES  None    None  None
9   emission_intensity_tco2e_per_mwh      DOUBLE  YES  None    None  None
10                    grid_connected     VARCHAR  YES  None    None  None
11                              grid     VARCHAR  YES  None    None  None
12                      primary_fuel  

In [29]:
print(con.execute("""
SELECT *
FROM nger_facility_emissions
LIMIT 10
""").fetchdf())

                 reporting_entity            facility_name facility_type  \
0  ACCIONA ENERGY OCEANIA PTY LTD        Gunning Wind Farm             F   
1  ACCIONA ENERGY OCEANIA PTY LTD       Royalla Solar Farm             F   
2  ACCIONA ENERGY OCEANIA PTY LTD         Waubra Wind Farm             F   
3  ACCIONA ENERGY OCEANIA PTY LTD          Corporate Total             C   
4              AGL ENERGY LIMITED         Banimboola Hydro             F   
5              AGL ENERGY LIMITED  Bayswater Power Station             F   
6              AGL ENERGY LIMITED     Bogong Power Station             F   
7              AGL ENERGY LIMITED         Burrendong Hydro             F   
8              AGL ENERGY LIMITED             Clover Hydro             F   
9              AGL ENERGY LIMITED     Coopers Cogeneration             F   

  state  electricity_production_gj  electricity_production_mwh  scope1_tco2e  \
0   NSW                   567719.0                    157700.0          19.0   
1  

In [30]:
print(con.execute("""
SELECT COUNT(*) AS total_rows
FROM nger_facility_emissions
""").fetchdf())

   total_rows
0        5321


In [31]:
con.execute("""
CREATE OR REPLACE TABLE nger_state_year_summary AS
SELECT
    TRIM(UPPER(state)) AS state,
    CAST(reporting_year AS INTEGER) AS year,
    SUM(electricity_production_mwh) AS total_generation_mwh,
    SUM(total_emissions_tco2e) AS total_emissions_tco2e
FROM nger_facility_emissions
WHERE facility_type = 'F'
  AND state IS NOT NULL
GROUP BY
    TRIM(UPPER(state)),
    CAST(reporting_year AS INTEGER)
""")

In [32]:
print(con.execute("DESCRIBE nger_state_year_summary").fetchdf())

             column_name column_type null   key default extra
0                  state     VARCHAR  YES  None    None  None
1                   year     INTEGER  YES  None    None  None
2   total_generation_mwh      DOUBLE  YES  None    None  None
3  total_emissions_tco2e      DOUBLE  YES  None    None  None


In [33]:
print(con.execute("""
SELECT state, year, COUNT(*) AS n
FROM nger_state_year_summary
GROUP BY state, year
ORDER BY state, year
""").fetchdf())

   state  year  n
0    ACT  2014  1
1    ACT  2015  1
2    ACT  2016  1
3    ACT  2017  1
4    ACT  2018  1
..   ...   ... ..
67    WA  2018  1
68    WA  2020  1
69    WA  2021  1
70    WA  2022  1
71    WA  2023  1

[72 rows x 3 columns]


In [34]:
print(con.execute("""
SELECT *
FROM nger_state_year_summary
ORDER BY state, year
LIMIT 20
""").fetchdf())

   state  year  total_generation_mwh  total_emissions_tco2e
0    ACT  2014               98106.0                 1708.0
1    ACT  2015               63256.0                 1788.0
2    ACT  2016               63949.0                 1956.0
3    ACT  2017               73003.0                 2355.0
4    ACT  2018               93555.0                 2061.0
5    ACT  2020               90253.0                 2979.0
6    ACT  2021               84427.0                 3786.0
7    ACT  2022              246566.0                 5287.0
8    ACT  2023              224556.0                 5621.0
9    NSW  2014            61897115.0             48539060.0
10   NSW  2015            67819197.0             51802322.0
11   NSW  2016            68317111.0             51462622.0
12   NSW  2017            68459457.0             52017521.0
13   NSW  2018            69041422.0             52407118.0
14   NSW  2020            66062106.0             47477514.0
15   NSW  2021            66059193.0    

In [35]:
print(con.execute("SHOW TABLES").fetchall())

[('abs_economy_industry',), ('nger_facility_emissions',), ('nger_state_year_summary',), ('renewable_projects_combined_standardized',)]


In [36]:
print(con.execute("""
SELECT state, MIN(year) AS min_year, MAX(year) AS max_year, COUNT(*) AS n_rows
FROM abs_economy_industry
GROUP BY state
ORDER BY state
""").fetchdf())

  state  min_year  max_year  n_rows
0   ACT      2020      2024       5
1   NSW      2020      2024       5
2    NT      2020      2024       5
3   QLD      2020      2024       5
4    SA      2020      2024       5
5   TAS      2020      2024       5
6   VIC      2020      2024       5
7    WA      2020      2024       5


In [37]:
print(con.execute("""
SELECT state, MIN(year) AS min_year, MAX(year) AS max_year, COUNT(*) AS n_rows
FROM nger_state_year_summary
GROUP BY state
ORDER BY state
""").fetchdf())

  state  min_year  max_year  n_rows
0   ACT      2014      2023       9
1   NSW      2014      2023       9
2    NT      2014      2023       9
3   QLD      2014      2023       9
4    SA      2014      2023       9
5   TAS      2014      2023       9
6   VIC      2014      2023       9
7    WA      2014      2023       9


In [38]:
print(con.execute("""
SELECT
    status,
    COUNT(*) AS total_rows,
    COUNT(project_date) AS non_null_project_date,
    COUNT(capacity_mw) AS non_null_capacity
FROM renewable_projects_combined_standardized
GROUP BY status
ORDER BY status
""").fetchdf())

print(con.execute("""
SELECT
    SUBSTR(project_date, 1, 4) AS year,
    COUNT(*) AS cnt
FROM renewable_projects_combined_standardized
WHERE project_date IS NOT NULL
GROUP BY SUBSTR(project_date, 1, 4)
ORDER BY year
""").fetchdf())

       status  total_rows  non_null_project_date  non_null_capacity
0  accredited          36                     36                 36
1   committed          36                     36                 36
2    probable          59                      0                 56
   year  cnt
0  2019    1
1  2021    1
2  2022    2
3  2023    3
4  2024   13
5  2025   44
6  2026    8


In [39]:
con.execute("""
CREATE OR REPLACE TABLE renewable_state_year_summary AS
SELECT
    state,
    CAST(SUBSTR(project_date, 1, 4) AS INTEGER) AS year,
    COUNT(*) AS renewable_project_count,
    SUM(capacity_mw) AS total_renewable_capacity_mw
FROM renewable_projects_combined_standardized
WHERE project_date IS NOT NULL
  AND state IS NOT NULL
GROUP BY
    state,
    CAST(SUBSTR(project_date, 1, 4) AS INTEGER)
""")

In [40]:
print(con.execute("DESCRIBE renewable_state_year_summary").fetchdf())

print(con.execute("""
SELECT state, year, COUNT(*) AS n
FROM renewable_state_year_summary
GROUP BY state, year
ORDER BY state, year
""").fetchdf())

print(con.execute("""
SELECT *
FROM renewable_state_year_summary
ORDER BY state, year
LIMIT 20
""").fetchdf())

                   column_name column_type null   key default extra
0                        state     VARCHAR  YES  None    None  None
1                         year     INTEGER  YES  None    None  None
2      renewable_project_count      BIGINT  YES  None    None  None
3  total_renewable_capacity_mw      DOUBLE  YES  None    None  None
   state  year  n
0    ACT  2025  1
1    NSW  2022  1
2    NSW  2023  1
3    NSW  2024  1
4    NSW  2025  1
5    NSW  2026  1
6    QLD  2023  1
7    QLD  2024  1
8    QLD  2025  1
9    QLD  2026  1
10    SA  2025  1
11    SA  2026  1
12   TAS  2025  1
13   TAS  2026  1
14   VIC  2021  1
15   VIC  2024  1
16   VIC  2025  1
17    WA  2019  1
18    WA  2024  1
19    WA  2025  1
20    WA  2026  1
   state  year  renewable_project_count  total_renewable_capacity_mw
0    ACT  2025                        1                        0.185
1    NSW  2022                        2                       65.000
2    NSW  2023                        2                  

In [41]:
con.execute("""
CREATE OR REPLACE TABLE integrated_energy_state_year AS
SELECT
    n.state,
    n.year,

    -- NGER
    n.total_generation_mwh,
    n.total_emissions_tco2e,

    -- ABS
    a.total_businesses,
    a.small_businesses,

    -- Renewable（可能为 NULL）
    r.renewable_project_count,
    r.total_renewable_capacity_mw

FROM nger_state_year_summary n

LEFT JOIN abs_economy_industry a
    ON n.state = a.state AND n.year = a.year

LEFT JOIN renewable_state_year_summary r
    ON n.state = r.state AND n.year = r.year

WHERE n.year BETWEEN 2020 AND 2023
""")

In [42]:
print(con.execute("DESCRIBE integrated_energy_state_year").fetchdf())

print(con.execute("""
SELECT state, year, COUNT(*) AS n
FROM integrated_energy_state_year
GROUP BY state, year
ORDER BY state, year
""").fetchdf())

print(con.execute("""
SELECT *
FROM integrated_energy_state_year
ORDER BY state, year
LIMIT 20
""").fetchdf())

                   column_name column_type null   key default extra
0                        state     VARCHAR  YES  None    None  None
1                         year     INTEGER  YES  None    None  None
2         total_generation_mwh      DOUBLE  YES  None    None  None
3        total_emissions_tco2e      DOUBLE  YES  None    None  None
4             total_businesses      BIGINT  YES  None    None  None
5             small_businesses      BIGINT  YES  None    None  None
6      renewable_project_count      BIGINT  YES  None    None  None
7  total_renewable_capacity_mw      DOUBLE  YES  None    None  None
   state  year  n
0    ACT  2020  1
1    ACT  2021  1
2    ACT  2022  1
3    ACT  2023  1
4    NSW  2020  1
5    NSW  2021  1
6    NSW  2022  1
7    NSW  2023  1
8     NT  2020  1
9     NT  2021  1
10    NT  2022  1
11    NT  2023  1
12   QLD  2020  1
13   QLD  2021  1
14   QLD  2022  1
15   QLD  2023  1
16    SA  2020  1
17    SA  2021  1
18    SA  2022  1
19    SA  2023  1
20   TAS  

In [43]:
print(con.execute("SHOW TABLES").fetchall())

[('abs_economy_industry',), ('integrated_energy_state_year',), ('nger_facility_emissions',), ('nger_state_year_summary',), ('renewable_projects_combined_standardized',), ('renewable_state_year_summary',)]


In [44]:
con.execute("""
COPY integrated_energy_state_year
TO 'integrated_energy_state_year.csv'
(HEADER, DELIMITER ',');
""")

In [45]:
print(con.execute("""
DESCRIBE renewable_projects_combined_standardized
""").fetchdf())

    column_name column_type null   key default extra
0  project_name     VARCHAR  YES  None    None  None
1         state     VARCHAR  YES  None    None  None
2      postcode      DOUBLE  YES  None    None  None
3   capacity_mw      DOUBLE  YES  None    None  None
4   fuel_source     VARCHAR  YES  None    None  None
5  project_date     VARCHAR  YES  None    None  None
6        status     VARCHAR  YES  None    None  None


In [46]:
print(con.execute("""
SELECT *
FROM renewable_projects_combined_standardized
LIMIT 10
""").fetchdf())

                                        project_name state  postcode  \
0                      DONS FORT - SOLAR W SGU - QLD   QLD    4660.0   
1                    Mintus Coles - Solar w SGU- NSW   NSW    2153.0   
2               Ventora - Acacia Ridge - Solar - QLD   QLD    4110.0   
3         Bayside Aged Care - Solar - Morisset - NSW   NSW    2264.0   
4  Matrix Composites and Engineering Pty Ltd - So...    WA    6166.0   
5                     Accensi Narangba - Solar - QLD   QLD    4504.0   
6                      Bunge Port Giles - Solar - SA    SA    5583.0   
7  HBF Stadium Carpark 4 PV System - Solar w SGU ...    WA    6010.0   
8          Stockland M-park Building C - Solar - NSW   NSW    2113.0   
9               Waverley Gardens - Solar w SGU - VIC   VIC    3170.0   

   capacity_mw fuel_source project_date      status  
0        0.421       Solar   2025-10-21  accredited  
1        0.190       Solar   2025-12-09  accredited  
2        0.162       Solar   2025-11-17  accr

In [47]:
import pandas as pd
import requests
import time
import re

source_df = con.execute("""
SELECT *
FROM renewable_projects_combined_standardized
""").fetchdf()

def clean_project_name(name):
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    
    # 去掉末尾州名
    name = re.sub(r"\s*-\s*(NSW|VIC|QLD|SA|WA|TAS|NT|ACT)\s*$", "", name, flags=re.IGNORECASE)
    
    # 去掉常见噪音词
    name = re.sub(r"\bSolar\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bw\s*SGU\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bSGU\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bPV System\b", "", name, flags=re.IGNORECASE)
    
    # 清理符号和多余空格
    name = re.sub(r"[-_/]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    return name

def build_query(row):
    parts = []
    
    clean_name = clean_project_name(row["project_name"])
    if pd.notna(clean_name) and clean_name:
        parts.append(clean_name)
        
    if pd.notna(row["state"]):
        parts.append(str(row["state"]).strip())
        
    if pd.notna(row["postcode"]):
        parts.append(str(int(row["postcode"])))
        
    parts.append("Australia")
    return ", ".join(parts)

source_df["clean_name"] = source_df["project_name"].apply(clean_project_name)
source_df["matched_query"] = source_df.apply(build_query, axis=1)

print(source_df[["project_name", "clean_name", "state", "postcode", "matched_query"]].head(10))

                                        project_name  \
0                      DONS FORT - SOLAR W SGU - QLD   
1                    Mintus Coles - Solar w SGU- NSW   
2               Ventora - Acacia Ridge - Solar - QLD   
3         Bayside Aged Care - Solar - Morisset - NSW   
4  Matrix Composites and Engineering Pty Ltd - So...   
5                     Accensi Narangba - Solar - QLD   
6                      Bunge Port Giles - Solar - SA   
7  HBF Stadium Carpark 4 PV System - Solar w SGU ...   
8          Stockland M-park Building C - Solar - NSW   
9               Waverley Gardens - Solar w SGU - VIC   

                                  clean_name state  postcode  \
0                                  DONS FORT   QLD    4660.0   
1                               Mintus Coles   NSW    2153.0   
2                       Ventora Acacia Ridge   QLD    4110.0   
3                 Bayside Aged Care Morisset   NSW    2264.0   
4  Matrix Composites and Engineering Pty Ltd    WA    6166.0   

In [48]:
def geocode_nominatim(query):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "jsonv2",
        "limit": 1,
        "countrycodes": "au"
    }
    headers = {
        "User-Agent": "COMP5339-assignment-geocoder/1.0"
    }
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=20)
        response.raise_for_status()
        data = response.json()
        
        if len(data) > 0:
            result = data[0]
            return {
                "latitude": float(result["lat"]),
                "longitude": float(result["lon"]),
                "display_name": result.get("display_name"),
                "geocode_status": "matched",
                "match_quality": "high"
            }
        else:
            return {
                "latitude": None,
                "longitude": None,
                "display_name": None,
                "geocode_status": "unmatched",
                "match_quality": "unmatched"
            }
    except Exception:
        return {
            "latitude": None,
            "longitude": None,
            "display_name": None,
            "geocode_status": "error",
            "match_quality": "unmatched"
        }

In [49]:
test_df = source_df.head(10).copy()

results = []
for query in test_df["matched_query"]:
    results.append(geocode_nominatim(query))
    time.sleep(1)   # Nominatim 公共服务最多 1 req/s

results_df = pd.DataFrame(results)
test_df = pd.concat([test_df.reset_index(drop=True), results_df], axis=1)

print(test_df[[
    "project_name",
    "matched_query",
    "latitude",
    "longitude",
    "display_name",
    "geocode_status"
]])

                                        project_name  \
0                      DONS FORT - SOLAR W SGU - QLD   
1                    Mintus Coles - Solar w SGU- NSW   
2               Ventora - Acacia Ridge - Solar - QLD   
3         Bayside Aged Care - Solar - Morisset - NSW   
4  Matrix Composites and Engineering Pty Ltd - So...   
5                     Accensi Narangba - Solar - QLD   
6                      Bunge Port Giles - Solar - SA   
7  HBF Stadium Carpark 4 PV System - Solar w SGU ...   
8          Stockland M-park Building C - Solar - NSW   
9               Waverley Gardens - Solar w SGU - VIC   

                                       matched_query   latitude   longitude  \
0                    DONS FORT, QLD, 4660, Australia        NaN         NaN   
1                 Mintus Coles, NSW, 2153, Australia        NaN         NaN   
2         Ventora Acacia Ridge, QLD, 4110, Australia        NaN         NaN   
3   Bayside Aged Care Morisset, NSW, 2264, Australia        NaN    

In [50]:
def clean_project_name_fallback(name):
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    
    # 去掉末尾州名
    name = re.sub(r"\s*-\s*(NSW|VIC|QLD|SA|WA|TAS|NT|ACT)\s*$", "", name, flags=re.IGNORECASE)
    
    # 去掉能源相关噪音词
    name = re.sub(r"\bSolar\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bw\s*SGU\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bSGU\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bPV System\b", "", name, flags=re.IGNORECASE)
    
    # 去掉企业/建筑类噪音
    name = re.sub(r"\bPty Ltd\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bBuilding [A-Z0-9]+\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bCarpark \d+\b", "", name, flags=re.IGNORECASE)
    
    # 清理连接符和多余空格
    name = re.sub(r"[-_/]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    return name


def make_short_name_fallback(name):
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    
    # 更激进一点，专门给 fallback 用
    name = re.sub(r"\bAged Care\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bPty Ltd\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bBuilding.*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bCarpark.*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bPV System\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bSolar\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bw\s*SGU\b", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\bSGU\b", "", name, flags=re.IGNORECASE)
    
    # 去掉末尾州名
    name = re.sub(r"\s*-\s*(NSW|VIC|QLD|SA|WA|TAS|NT|ACT)\s*$", "", name, flags=re.IGNORECASE)
    
    # 统一符号
    name = re.sub(r"[-_/]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    return name

In [51]:
source_fallback_df = source_df.copy()

source_fallback_df["clean_name_fallback"] = source_fallback_df["project_name"].apply(clean_project_name_fallback)
source_fallback_df["short_name_fallback"] = source_fallback_df["project_name"].apply(make_short_name_fallback)

print(source_fallback_df[[
    "project_name",
    "clean_name_fallback",
    "short_name_fallback"
]].head(10))

                                        project_name  \
0                      DONS FORT - SOLAR W SGU - QLD   
1                    Mintus Coles - Solar w SGU- NSW   
2               Ventora - Acacia Ridge - Solar - QLD   
3         Bayside Aged Care - Solar - Morisset - NSW   
4  Matrix Composites and Engineering Pty Ltd - So...   
5                     Accensi Narangba - Solar - QLD   
6                      Bunge Port Giles - Solar - SA   
7  HBF Stadium Carpark 4 PV System - Solar w SGU ...   
8          Stockland M-park Building C - Solar - NSW   
9               Waverley Gardens - Solar w SGU - VIC   

                 clean_name_fallback                short_name_fallback  
0                          DONS FORT                          DONS FORT  
1                       Mintus Coles                       Mintus Coles  
2               Ventora Acacia Ridge               Ventora Acacia Ridge  
3         Bayside Aged Care Morisset                   Bayside Morisset  
4  Matrix Com

In [52]:
def geocode_nominatim_with_fallback(row):
    queries = []
    
    clean_name = row["clean_name_fallback"]
    short_name = row["short_name_fallback"]
    state = row["state"]
    postcode = row["postcode"]
    
    # 1. 最完整：clean_name + state + postcode + Australia
    if pd.notna(clean_name) and clean_name and pd.notna(postcode):
        queries.append({
            "query": f"{clean_name}, {state}, {int(postcode)}, Australia",
            "match_quality": "high"
        })
    
    # 2. 中等：clean_name + state + Australia
    if pd.notna(clean_name) and clean_name:
        queries.append({
            "query": f"{clean_name}, {state}, Australia",
            "match_quality": "medium"
        })
    
    # 3. 最弱：short_name + state + Australia
    if pd.notna(short_name) and short_name:
        queries.append({
            "query": f"{short_name}, {state}, Australia",
            "match_quality": "low"
        })
    
    for item in queries:
        result = geocode_nominatim(item["query"])
        time.sleep(1)   # 遵守 Nominatim 公共服务速率要求
        
        if result["latitude"] is not None:
            return {
                "nominatim_fallback_query": item["query"],
                "nominatim_fallback_latitude": result["latitude"],
                "nominatim_fallback_longitude": result["longitude"],
                "nominatim_fallback_display_name": result["display_name"],
                "nominatim_fallback_status": result["geocode_status"],
                "nominatim_fallback_match_quality": item["match_quality"]
            }
    
    return {
        "nominatim_fallback_query": queries[0]["query"] if len(queries) > 0 else None,
        "nominatim_fallback_latitude": None,
        "nominatim_fallback_longitude": None,
        "nominatim_fallback_display_name": None,
        "nominatim_fallback_status": "unmatched",
        "nominatim_fallback_match_quality": "unmatched"
    }

In [53]:
nominatim_fallback_test_df = source_fallback_df.head(10).copy()

nominatim_fallback_test_results = []
for _, row in nominatim_fallback_test_df.iterrows():
    nominatim_fallback_test_results.append(geocode_nominatim_with_fallback(row))

nominatim_fallback_test_results_df = pd.DataFrame(nominatim_fallback_test_results)

nominatim_fallback_test_df = pd.concat(
    [nominatim_fallback_test_df.reset_index(drop=True), nominatim_fallback_test_results_df],
    axis=1
)

print(nominatim_fallback_test_df[[
    "project_name",
    "clean_name_fallback",
    "short_name_fallback",
    "nominatim_fallback_query",
    "nominatim_fallback_latitude",
    "nominatim_fallback_longitude",
    "nominatim_fallback_display_name",
    "nominatim_fallback_status",
    "nominatim_fallback_match_quality"
]])

                                        project_name  \
0                      DONS FORT - SOLAR W SGU - QLD   
1                    Mintus Coles - Solar w SGU- NSW   
2               Ventora - Acacia Ridge - Solar - QLD   
3         Bayside Aged Care - Solar - Morisset - NSW   
4  Matrix Composites and Engineering Pty Ltd - So...   
5                     Accensi Narangba - Solar - QLD   
6                      Bunge Port Giles - Solar - SA   
7  HBF Stadium Carpark 4 PV System - Solar w SGU ...   
8          Stockland M-park Building C - Solar - NSW   
9               Waverley Gardens - Solar w SGU - VIC   

                 clean_name_fallback                short_name_fallback  \
0                          DONS FORT                          DONS FORT   
1                       Mintus Coles                       Mintus Coles   
2               Ventora Acacia Ridge               Ventora Acacia Ridge   
3         Bayside Aged Care Morisset                   Bayside Morisset   
4  Matri

In [54]:
nominatim_fallback_results = []

for i, (_, row) in enumerate(source_fallback_df.iterrows(), start=1):
    result = geocode_nominatim_with_fallback(row)
    nominatim_fallback_results.append(result)
    
    if i % 10 == 0:
        print(f"Processed {i} rows...")

Processed 10 rows...
Processed 20 rows...
Processed 30 rows...
Processed 40 rows...
Processed 50 rows...
Processed 60 rows...
Processed 70 rows...
Processed 80 rows...
Processed 90 rows...
Processed 100 rows...
Processed 110 rows...
Processed 120 rows...
Processed 130 rows...


In [55]:
nominatim_fallback_results_df = pd.DataFrame(nominatim_fallback_results)

renewable_projects_geocoded_nominatim_fallback_df = pd.concat(
    [source_fallback_df.reset_index(drop=True), nominatim_fallback_results_df],
    axis=1
)

print(renewable_projects_geocoded_nominatim_fallback_df.head())

                                        project_name state  postcode  \
0                      DONS FORT - SOLAR W SGU - QLD   QLD    4660.0   
1                    Mintus Coles - Solar w SGU- NSW   NSW    2153.0   
2               Ventora - Acacia Ridge - Solar - QLD   QLD    4110.0   
3         Bayside Aged Care - Solar - Morisset - NSW   NSW    2264.0   
4  Matrix Composites and Engineering Pty Ltd - So...    WA    6166.0   

   capacity_mw fuel_source project_date      status  \
0        0.421       Solar   2025-10-21  accredited   
1        0.190       Solar   2025-12-09  accredited   
2        0.162       Solar   2025-11-17  accredited   
3        0.140       Solar   2025-12-16  accredited   
4        0.850       Solar   2025-12-17  accredited   

                                  clean_name  \
0                                  DONS FORT   
1                               Mintus Coles   
2                       Ventora Acacia Ridge   
3                 Bayside Aged Care Morisset

In [56]:
total_rows = len(renewable_projects_geocoded_nominatim_fallback_df)
matched_rows = renewable_projects_geocoded_nominatim_fallback_df["nominatim_fallback_latitude"].notna().sum()
match_rate = matched_rows / total_rows

print("Total rows:", total_rows)
print("Matched rows:", matched_rows)
print("Match rate:", round(match_rate, 3))

Total rows: 131
Matched rows: 27
Match rate: 0.206


In [57]:
match_quality_summary = (
    renewable_projects_geocoded_nominatim_fallback_df["nominatim_fallback_match_quality"]
    .value_counts(dropna=False)
    .reset_index()
)
match_quality_summary.columns = ["match_quality", "count"]

print(match_quality_summary)

  match_quality  count
0     unmatched    104
1          high     14
2        medium     12
3           low      1


In [58]:
status_summary = (
    renewable_projects_geocoded_nominatim_fallback_df
    .assign(is_matched=renewable_projects_geocoded_nominatim_fallback_df["nominatim_fallback_latitude"].notna())
    .groupby("status")
    .agg(
        total_rows=("status", "size"),
        matched_rows=("is_matched", "sum")
    )
    .reset_index()
)

status_summary["match_rate"] = (status_summary["matched_rows"] / status_summary["total_rows"]).round(3)

print(status_summary)

       status  total_rows  matched_rows  match_rate
0  accredited          36            15       0.417
1   committed          36             4       0.111
2    probable          59             8       0.136


In [59]:
print(
    renewable_projects_geocoded_nominatim_fallback_df[
        renewable_projects_geocoded_nominatim_fallback_df["nominatim_fallback_latitude"].notna()
    ][[
        "project_name",
        "nominatim_fallback_query",
        "nominatim_fallback_display_name",
        "nominatim_fallback_match_quality"
    ]].head(20)
)

                                         project_name  \
3          Bayside Aged Care - Solar - Morisset - NSW   
5                      Accensi Narangba - Solar - QLD   
7   HBF Stadium Carpark 4 PV System - Solar w SGU ...   
9                Waverley Gardens - Solar w SGU - VIC   
10                   The Barracks - Solar w SGU - QLD   
12                                  Tabbita-Solar-NSW   
14                 BAULKHAM HILLS - SOLAR W SGU - NSW   
20                       40 Cameron Ave - Solar - ACT   
21                         BC Westcourt - Solar - QLD   
26                   BC Parkinson - Solar w SGU - QLD   
27                   BC Sunnybank Hills - Solar - QLD   
31                    BC Murrumba Downs - Solar - QLD   
33                               Glen Innes-Solar-NSW   
34  Tweed City Shopping Centre, Tweed Heads South ...   
35                         BC Bundaberg - Solar - QLD   
39                           Moorebank Logistics Park   
49                             

In [60]:
con.register("geo_view", renewable_projects_geocoded_nominatim_fallback_df)

con.execute("""
CREATE OR REPLACE TABLE renewable_projects_geocoded_nominatim_fallback AS
SELECT * FROM geo_view
""")

In [61]:

print(con.execute("SHOW TABLES").fetchall())

[('abs_economy_industry',), ('geo_view',), ('integrated_energy_state_year',), ('nger_facility_emissions',), ('nger_state_year_summary',), ('renewable_projects_combined_standardized',), ('renewable_projects_geocoded_nominatim_fallback',), ('renewable_state_year_summary',)]


In [62]:
con.execute("DROP VIEW geo_view")

In [63]:
print(con.execute("SHOW TABLES").fetchall())

[('abs_economy_industry',), ('integrated_energy_state_year',), ('nger_facility_emissions',), ('nger_state_year_summary',), ('renewable_projects_combined_standardized',), ('renewable_projects_geocoded_nominatim_fallback',), ('renewable_state_year_summary',)]
